# Copilot adoption journey and ways-of-working analysis

This end-to-end companion to `copilot-analytics-examples.ipynb` uses a Viva Insights Person Query
to move from basic Copilot metrics to an adoption journey and ways-of-working assessment.
It answers four practical questions:

1. **Reach:** how the licensed Copilot population (where available), metric coverage,
   and weekly activation expanded over time.
2. **Habit:** how many people show sustained use, emerging use, or no use, using Power
   and Habitual User status as the notebook's proxy for *stickiness* - durable, repeated
   return usage rather than a one-off trial.
3. **Opportunity:** where adoption differs across managers, individual contributors, and functions.
4. **Ways of working:** how sustained use is associated with collaboration load, meetings,
   focus, multitasking, and after-hours work.

The analysis is observational. It describes associations and adoption patterns; it does not
claim that Copilot caused the working-pattern differences.

The notebook is organization-agnostic: it makes no assumptions about a specific customer's
org structure, headcount, or function names. Set `INPUT_FILE` to a local Person Query
export before running it. The export should contain at least 12 weeks of Copilot activity
and the collaboration/work-pattern metrics used by the optional diagnostic sections.


In [ ]:
from pathlib import Path

INPUT_FILE = Path("person_query.parquet")  # Replace with your local Person Query export
OUTPUT_DIR = Path("outputs/copilot-adoption-journey")

MIN_PRIVACY_N = 5
MIN_DISPLAY_N = 30
POWER_THRESHOLD = 15
SEGMENT_VERSION = "12w"
SEGMENT_WINDOW_WEEKS = 12
RECENT_WEEKS = 4

# Illustrative Novice -> Power/Habitual conversion rates for the "sizing the prize"
# scenarios, in addition to the internal manager-benchmark rate computed from the data.
NOVICE_CONVERSION_RATES = [0.25, 0.40]

# Functions at or above this Power + Habitual adoption percentage are treated as
# "already working" and surfaced as a replication playbook.
LEADING_FUNCTION_THRESHOLD_PCT = 50.0

# Location values known to be data artifacts (e.g. a registered/HQ mailing address used
# as a default rather than a genuine work-site cohort). Excluded only from the Location
# cut of create_rank(); people keep their FunctionType and ManagerStatus rows. Leave empty
# unless a customer-specific artifact has been confirmed.
LOCATION_EXCLUDE = []

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Input:  {INPUT_FILE.resolve()}")
print(f"Output: {OUTPUT_DIR.resolve()}")

## 1. Method and interpretation guardrails

- One row should represent one person-week.
- Blank rows are removed before analysis.
- When `Copilot_enabled_days` is present, it is used for the licensed-population view;
  otherwise, non-null Copilot actions/active-days values are treated as **metric coverage**,
  not asserted to be confirmed licensing.
- Usage segmentation uses the standard **12-week rolling definition** documented in the
  [`vivainsights::identify_usage_segments()` reference](https://microsoft.github.io/vivainsights/reference/identify_usage_segments.html).
- The first weeks of any panel have incomplete rolling histories. Later coverage cohorts
  may not yet have enough observed weeks to qualify as Habitual or Power Users.
- Groups smaller than 30 are excluded from function comparisons, above the privacy floor of 5.
- Adjusted comparisons control for function and manager status when those attributes are
  available, but cannot remove all role or seniority differences.

In [ ]:
import re
import warnings

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import vivainsights as vi

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 80)

C_NAVY = "#16324F"
C_BLUE = "#2F6B9A"
C_TEAL = "#2A7F83"
C_GOLD = "#B9892D"
C_RED = "#A4473D"
C_GREY = "#7A8288"
C_LIGHT = "#E8EDF1"
C_TEXT = "#202428"

plt.rcParams.update({
    "font.family": ["Segoe UI", "DejaVu Sans", "sans-serif"],
    "font.size": 10.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "text.color": C_TEXT,
    "figure.dpi": 120,
})

TABLES = {}
FIGURES = {}


def save_table(frame, name):
    frame.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)
    TABLES[name] = frame.copy()
    return frame


def save_figure(fig, name):
    fig.savefig(OUTPUT_DIR / f"{name}.png", dpi=180, bbox_inches="tight", facecolor="white")
    FIGURES[name] = fig
    return fig


def add_subtitle(ax, text):
    """Add a small italic subtitle stating the exact metric shown, directly under an
    axis title. Charts without an explicit metric named in the title read ambiguously
    once separated from the surrounding narrative text (e.g. on a slide); this keeps
    the metric definition attached to the chart itself. Placed comfortably clear of
    the title line (1.14 axes-fraction) to avoid glyph overlap with bold titles that
    have descenders/ascenders, especially once the figure is scaled down on a slide."""
    ax.text(0.0, 1.14, text, transform=ax.transAxes, fontsize=8.5,
            style="italic", color=C_GREY, ha="left", va="bottom")


def pct(num, den):
    return np.nan if not den else 100 * num / den


def normalise_column(name):
    return re.sub(r"[^0-9A-Za-z]+", "_", str(name)).strip("_")

In [ ]:
raw = pd.read_parquet(INPUT_FILE)
raw_rows = len(raw)
raw.columns = [normalise_column(c) for c in raw.columns]

action_columns = [
    col for col in raw.columns
    if col.startswith("Copilot_actions_taken_in")
]
if "Total_Copilot_actions_taken" not in raw.columns and action_columns:
    raw[action_columns] = raw[action_columns].fillna(0)
    raw["Total_Copilot_actions_taken"] = raw[action_columns].sum(axis=1)

required = {"PersonId", "MetricDate", "Total_Copilot_actions_taken",
            "Total_Copilot_active_days"}
missing = sorted(required - set(raw.columns))
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = raw.dropna(subset=["PersonId", "MetricDate"]).copy()
df["MetricDate"] = pd.to_datetime(df["MetricDate"], errors="coerce")
df = df.dropna(subset=["MetricDate"])

bad_text = {"", "NA", "NULL", "#N/A", "#n/a", "N/A"}
for col in ["Organization", "FunctionType", "LevelDesignation", "Location"]:
    if col in df:
        df[col] = df[col].mask(df[col].astype(str).isin(bad_text))

def normalise_manager_status(value):
    if pd.isna(value):
        return "Unknown"
    text = str(value).strip().upper()
    if text in {"MANAGER", "YES", "TRUE", "Y", "1"}:
        return "Manager"
    if text in {"IC", "INDIVIDUAL CONTRIBUTOR", "NO", "FALSE", "N", "0"}:
        return "IC"
    return str(value).strip()

df["ManagerStatus"] = (
    df["IsManager"].map(normalise_manager_status)
    if "IsManager" in df.columns
    else "Unknown"
)
for optional_col in ["FunctionType", "Location", "LevelDesignation"]:
    if optional_col not in df.columns:
        df[optional_col] = "Unknown"

duplicate_person_weeks = int(df.duplicated(["PersonId", "MetricDate"]).sum())
if duplicate_person_weeks:
    raise ValueError(f"Found {duplicate_person_weeks:,} duplicate person-week rows.")

df = df.sort_values(["PersonId", "MetricDate"]).reset_index(drop=True)
df["copilot_observed"] = (
    df["Total_Copilot_actions_taken"].notna()
    & df["Total_Copilot_active_days"].notna()
)
df["copilot_active"] = df["Total_Copilot_actions_taken"].fillna(0) > 0
# Distinct from `copilot_active` (actions-based): this flag is used for the
# population/licensing/active-use chart series, per Total_Copilot_active_days.
df["copilot_active_days_flag"] = df["Total_Copilot_active_days"].fillna(0) > 0

# Licensing detection. `Copilot_enabled_days` is the standard Viva Insights field for a
# confirmed licence day-count, but it is not present in every Person Query export (it is
# absent from this one). Where it exists, licensing is measured directly as
# n_distinct(PersonId) with Copilot_enabled_days > 0 per MetricDate; otherwise the notebook
# keeps relying on non-null Copilot telemetry (`copilot_observed`) as the best available
# proxy, per the Section 1 guardrails. This keeps the notebook organization-agnostic across
# customers whose exports do and do not include the field.
LICENSE_COL = "Copilot_enabled_days"
has_license_col = LICENSE_COL in df.columns
if has_license_col:
    df[LICENSE_COL] = pd.to_numeric(df[LICENSE_COL], errors="coerce")
    df["copilot_licensed"] = df[LICENSE_COL] > 0
else:
    df["copilot_licensed"] = np.nan

blank_rows_removed = raw_rows - len(df)
weeks = np.sort(df["MetricDate"].unique())
latest_week = pd.Timestamp(weeks[-1])


# Report-wide metadata used to build informative footnotes across every deliverable
# (Word report and PowerPoint deck): date range covered, and population size broken
# down by total measured population, Copilot-licensed population, and Copilot-active
# population, all as of the latest observed week.
total_population_n = int(df.loc[df["MetricDate"] == weeks[-1], "PersonId"].nunique())
licensed_population_n = int(
    df.loc[(df["MetricDate"] == weeks[-1]) & df["copilot_licensed"].fillna(True), "PersonId"].nunique()
) if has_license_col else total_population_n
active_population_n = int(
    df.loc[(df["MetricDate"] == weeks[-1]) & df["copilot_active_days_flag"], "PersonId"].nunique()
)
report_metadata = pd.DataFrame([{
    "analysis_start": pd.Timestamp(weeks[0]).date().isoformat(),
    "analysis_end": pd.Timestamp(weeks[-1]).date().isoformat(),
    "weeks_covered": len(weeks),
    "total_population_n": total_population_n,
    "licensed_population_n": licensed_population_n,
    "active_population_n": active_population_n,
    "licensed_is_assumed": not has_license_col,
}])
save_table(report_metadata, "00_report_metadata")
FOOTNOTE_BASE = (
    f"{report_metadata.at[0, 'analysis_start']} to {report_metadata.at[0, 'analysis_end']} "
    f"({len(weeks)} weeks) | n={total_population_n:,} total, "
    f"{licensed_population_n:,} licensed"
    + ("*" if not has_license_col else "")
    + f", {active_population_n:,} active in latest week"
)
print(f"Footnote base string: {FOOTNOTE_BASE}")

print(f"Rows in file:          {raw_rows:,}")
print(f"Blank rows removed:    {blank_rows_removed:,}")
print(f"Analysis rows:         {len(df):,}")
print(f"People:                {df['PersonId'].nunique():,}")
print(f"Weeks:                 {df['MetricDate'].nunique()} "
      f"({df['MetricDate'].min().date()} to {latest_week.date()})")
print(f"Duplicate person-weeks:{duplicate_person_weeks:,}")
print(f"LevelDesignation coverage: {df['LevelDesignation'].notna().mean() * 100:.1f}%")
print(f"Copilot_enabled_days present:  {has_license_col}")

## 2. Coverage and activation momentum

The null pattern in the Copilot metrics changes over time. Where the data supports it,
the notebook keeps up to three separate measures, from broadest to narrowest:

- **Licensed population** *(shown only when `Copilot_enabled_days` is present in the
  export)*: people with at least one enabled day in the week, counted as
  `n_distinct(PersonId)` grouped by `Copilot_enabled_days > 0` over `MetricDate`. This is
  the earliest, broadest lens - it reflects provisioning, not usage.
- **Metric coverage:** people with a non-null Copilot actions and active-days value. When
  `Copilot_enabled_days` is absent, this is the best available proxy for licensing.
- **Weekly activation:** covered people with at least one Copilot action.

This distinction matters because a rise in covered people may reflect licence rollout,
query scope, or telemetry completeness. It should not automatically be labelled adoption.

In [ ]:
weekly = (
    df.groupby("MetricDate")
      .agg(
          population=("PersonId", "nunique"),
          copilot_covered=("copilot_observed", "sum"),
          active=("copilot_active", "sum"),
          active_days_covered=("copilot_active_days_flag", "sum"),
          licensed=("copilot_licensed", "sum"),
          total_actions=("Total_Copilot_actions_taken", "sum"),
      )
      .reset_index()
)
weekly["coverage_pct"] = 100 * weekly["copilot_covered"] / weekly["population"]
weekly["active_pct_covered"] = 100 * weekly["active"] / weekly["copilot_covered"]
weekly["active_pct_population"] = 100 * weekly["active"] / weekly["population"]
weekly["actions_per_active"] = weekly["total_actions"] / weekly["active"]
weekly["active_days_covered_pct"] = 100 * weekly["active_days_covered"] / weekly["population"]
if has_license_col:
    weekly["licensed_pct"] = 100 * weekly["licensed"] / weekly["population"]
else:
    # No enabled-days field is available for this export. Rather than dropping the
    # licensed tier from the chart entirely, show it as an assumed upper-bound equal to
    # the total measured population (the most defensible assumption absent contrary
    # evidence), clearly styled and labelled as an assumption so it cannot be mistaken
    # for confirmed licensing. This keeps the three-tier population/licensing/activation
    # story visually consistent across every client, whether or not enabled-days exists.
    weekly["licensed"] = weekly["population"]
    weekly["licensed_pct"] = 100.0

observed = df[df["copilot_observed"]].copy()
first_observed = observed.groupby("PersonId")["MetricDate"].min().rename("first_observed")
newly_observed = first_observed.value_counts().sort_index().rename("newly_observed")
weekly = weekly.merge(newly_observed, left_on="MetricDate", right_index=True, how="left")
weekly["newly_observed"] = weekly["newly_observed"].fillna(0).astype(int)
save_table(weekly, "01_weekly_coverage_and_activation")

# Three side-by-side single-axis panels rather than a two-panel design with a secondary
# (twin) axis - a dual-axis chart invites misreading, since the two y-scales are easy to
# conflate at a glance.
fig, axes = plt.subplots(1, 3, figsize=(19, 4.8))

ax = axes[0]
ax.plot(weekly["MetricDate"], weekly["population"], marker="o", lw=2.0,
        color=C_GREY, linestyle="--", label="Total population (Viva Insights)")
if has_license_col:
    ax.plot(weekly["MetricDate"], weekly["licensed"], marker="o", lw=2.3,
            color=C_GOLD, label="Licensed (Copilot_enabled_days > 0)")
else:
    ax.plot(weekly["MetricDate"], weekly["licensed"], lw=2.3, linestyle=":",
            color=C_GOLD, label="Licensed (assumed = full population*)")
ax.plot(weekly["MetricDate"], weekly["active_days_covered"], marker="o", lw=2.3,
        color=C_TEAL, label="Active (Copilot_active_days > 0)")
ax.set_ylabel("People")
ax.set_ylim(0, weekly["population"].max() * 1.08)
ax.legend(frameon=False, fontsize=8.5, loc="lower left")
ax.set_title("Population, licensing, and active use over time", loc="left",
             fontweight="bold")
add_subtitle(ax, "n_distinct(PersonId) by MetricDate | population, licensed (Copilot_enabled_days > 0), active (Copilot_active_days > 0)")
if not has_license_col:
    ax.annotate(
        "*Copilot_enabled_days is unavailable; licensed is shown as an assumed\n"
        "upper bound equal to total population, not confirmed licensing.",
        xy=(0.01, -0.32), xycoords="axes fraction", fontsize=7.5, color=C_GREY,
    )

ax = axes[1]
new_after_baseline = weekly["newly_observed"].copy()
new_after_baseline.iloc[0] = 0
ax.bar(weekly["MetricDate"], new_after_baseline, width=5.2, color=C_GOLD,
       label="Newly covered")
ax.set_ylabel("Newly covered people")
ax.set_title("New coverage arrived in bursts", loc="left", fontweight="bold")
add_subtitle(ax, "Count of people with a first-observed Copilot metric date in that week")
ax.grid(axis="x", visible=False)

ax = axes[2]
ax.plot(weekly["MetricDate"], weekly["actions_per_active"], color=C_BLUE,
        marker="o", lw=2.2, label="Actions per active user")
ax.set_ylabel("Actions per active user")
ax.set_title("Usage depth stayed comparatively stable", loc="left", fontweight="bold")
add_subtitle(ax, "Mean total Copilot actions taken per active person, per week")

for ax in axes:
    ax.tick_params(axis="x", rotation=30)

# No organization-name prefix: the chart is always shown under an organization-branded
# slide or section heading, so baking a config string into the figure title risks an
# awkward, mid-sentence-looking title when ORG_LABEL is left at its generic default.
fig.suptitle("Copilot population, licensing, and activation", x=0.01,
             ha="left", fontsize=15, fontweight="bold")
fig.tight_layout()
save_figure(fig, "01_coverage_and_activation")
plt.show()

first_row, last_row = weekly.iloc[0], weekly.iloc[-1]
largest_additions = weekly.iloc[1:].nlargest(3, "newly_observed")
print(
    f"Metric coverage moved from {first_row['coverage_pct']:.1f}% to "
    f"{last_row['coverage_pct']:.1f}%; active among covered moved from "
    f"{first_row['active_pct_covered']:.1f}% to {last_row['active_pct_covered']:.1f}%."
)
if has_license_col:
    print(
        f"Licensed population moved from {first_row['licensed_pct']:.1f}% to "
        f"{last_row['licensed_pct']:.1f}% of the measured population."
    )
else:
    print(
        "Copilot_enabled_days is not present in this export. The licensed series is "
        "shown as an assumed upper bound equal to the total population (dotted line), "
        "not confirmed licensing."
    )
print(
    f"Actions per active user were {first_row['actions_per_active']:.1f} initially and "
    f"{last_row['actions_per_active']:.1f} in the latest week."
)
print("\nLargest additions to Copilot metric coverage after the baseline week:")
print(largest_additions[["MetricDate", "newly_observed", "coverage_pct",
                         "active_pct_covered"]].to_string(index=False))

## 3. Adoption journey and habit formation

**Why this section matters:** a single week of use does not tell you whether Copilot has
become part of someone's routine. Power and Habitual User status is this notebook's proxy
for **stickiness** - evidence of durable, repeated return usage rather than a one-off
trial. Both segments require at least one action in **9 of the trailing 12 weeks**, so
membership cannot be earned by a single busy week; it requires activity spread across
roughly a quarter. Power Users add a volume threshold on top of that consistency, which
separates heavy, embedded use from lighter-but-still-durable habitual use. Together they
answer a question a simple "used it this week" metric cannot: has the behaviour persisted
long enough to call it a habit, and is it a return habit rather than a trial?

The notebook applies the authoritative **12-week rolling usage-segment definition** from
[`identify_usage_segments()`](https://microsoft.github.io/vivainsights/reference/identify_usage_segments.html).
An "active week" means the target metric records at least one action.

- **Power User:** at least one action in 9 or more of the trailing 12 weeks **and**
  an average of at least 15 weekly actions over the rolling period. The R reference exposes
  this threshold through `power_thres`; the Python 0.4.3 preset implementation currently
  applies 15 directly.
- **Habitual User:** at least one action in 9 or more of the trailing 12 weeks, but the
  rolling weekly average is below the Power User threshold.
- **Novice User:** a rolling average of at least one weekly action, without meeting the
  9-of-12 habit requirement.
- **Low User:** at least one action during the rolling period, but an average below one
  weekly action and without meeting the habit requirement.
- **Non-user:** no actions during the rolling period.

The categories are evaluated in that order, so Power Users are a high-volume subset of
habitual users. Because the package calculates rolling averages from available history,
Novice, Low, and Non-user labels can appear before a person has 12 observed weeks. A person
cannot satisfy the 9-of-12 Habitual or Power requirement without at least nine active weeks.

For executive interpretation, Power and Habitual are combined as **Power + Habitual
Users**, Novice and Low as **Emerging**, with Non-user retained separately. A plain-English glossary
of every segment, saved as `00_usage_segment_definitions`, is exported below as a
standalone asset so the definitions can be dropped directly into decks and reports.

In [ ]:
segment_definitions = pd.DataFrame([
    {
        "segment": "Power User",
        "definition": (
            f"Active in at least 9 of the trailing 12 weeks AND an average of at least "
            f"{POWER_THRESHOLD} weekly actions over that window."
        ),
        "what_it_indicates": (
            "The highest-stickiness usage: frequent return visits at high volume. The "
            "clearest evidence that Copilot has become embedded in someone's workflow."
        ),
    },
    {
        "segment": "Habitual User",
        "definition": (
            "Active in at least 9 of the trailing 12 weeks, with a rolling weekly average "
            "below the Power User threshold."
        ),
        "what_it_indicates": (
            "A durable weekly habit has formed even though usage volume is modest - "
            "consistency rather than intensity is the signal."
        ),
    },
    {
        "segment": "Novice User",
        "definition": (
            "A rolling average of at least one weekly action, without meeting the 9-of-12 "
            "week requirement."
        ),
        "what_it_indicates": (
            "Has tried Copilot repeatedly but has not yet formed a consistent weekly habit "
            "- the main pool for conversion into Power + Habitual use."
        ),
    },
    {
        "segment": "Low User",
        "definition": (
            "At least one action in the rolling period, but a rolling average below one "
            "weekly action and not meeting the habit requirement."
        ),
        "what_it_indicates": "Sporadic, occasional use only.",
    },
    {
        "segment": "Non-user",
        "definition": "No Copilot actions recorded during the rolling 12-week period.",
        "what_it_indicates": "No observed usage.",
    },
])
save_table(segment_definitions, "00_usage_segment_definitions")
print(segment_definitions.to_string(index=False))

In [ ]:
# Preserve calendar weeks after first observed Copilot telemetry. Dropping
# null rows would compress a 12-week rolling window into the last 12 observed rows.
seg_input = df.merge(first_observed, on="PersonId", how="inner")
seg_input = seg_input[seg_input["MetricDate"] >= seg_input["first_observed"]].copy()
seg_input["Total_Copilot_actions_taken"] = (
    seg_input["Total_Copilot_actions_taken"].fillna(0).astype(float)
)
seg = vi.identify_usage_segments(
    seg_input,
    metric="Total_Copilot_actions_taken",
    version=SEGMENT_VERSION,
    power_thres=POWER_THRESHOLD,
    return_type="data",
)
source_segment_col = f"UsageSegments_{SEGMENT_VERSION}"
seg = seg.rename(columns={source_segment_col: "UsageSegment_12w"})

latest_segments = (
    seg[seg["MetricDate"] == latest_week][["PersonId", "UsageSegment_12w"]]
    .drop_duplicates("PersonId")
)
journey_map = {
    "Power User": "Power + Habitual",
    "Habitual User": "Power + Habitual",
    "Novice User": "Emerging",
    "Low User": "Emerging",
    "Non-user": "Non-user",
}
latest_segments["JourneyStage"] = latest_segments["UsageSegment_12w"].map(journey_map)

segment_order = ["Power User", "Habitual User", "Novice User", "Low User", "Non-user"]
# Matches vivainsights.identify_usage_segments()'s own default plot colours exactly
# (see identify_usage_segments.py: category_order + colors), so every chart in this
# notebook is visually consistent with the package's native segment chart.
# vivainsights.identify_usage_segments()'s own default plot colours give Low User
# ("#808080") and Non-user ("grey") the same rendered grey, which is indistinguishable
# on a slide. We keep Power/Habitual/Novice identical to the native palette but assign
# Low User a distinct gold tone so all five segments remain visually separable.
segment_color_map = {
    "Power User": "#0c336e",
    "Habitual User": "#1c66b0",
    "Novice User": "#80baea",
    "Low User": "#B9892D",
    "Non-user": "#808080",
}
segment_colors = [segment_color_map[segment] for segment in segment_order]

# A single, shared colour scheme for the 3-stage journey grouping (Power + Habitual /
# Emerging / Non-user), reused consistently across every chart that shows it.
journey_stage_order = ["Power + Habitual", "Emerging", "Non-user"]
journey_stage_color_map = {
    "Power + Habitual": segment_color_map["Power User"],
    "Emerging": segment_color_map["Novice User"],
    "Non-user": segment_color_map["Non-user"],
}
segment_summary = (
    latest_segments["UsageSegment_12w"].value_counts()
    .reindex(segment_order, fill_value=0)
    .rename_axis("segment").reset_index(name="people")
)
segment_summary["pct"] = 100 * segment_summary["people"] / segment_summary["people"].sum()
save_table(segment_summary, "02_latest_usage_segments")

person_journey = (
    observed.groupby("PersonId")
    .agg(
        first_observed=("MetricDate", "min"),
        observed_weeks=("MetricDate", "nunique"),
        active_weeks=("copilot_active", "sum"),
        total_actions=("Total_Copilot_actions_taken", "sum"),
    )
    .reset_index()
    .merge(latest_segments, on="PersonId", how="left")
)
person_journey["active_share"] = (
    person_journey["active_weeks"] / person_journey["observed_weeks"]
)

cohort_latest = (
    person_journey.groupby("first_observed")
    .agg(
        people=("PersonId", "size"),
        active_latest=("PersonId", lambda ids: int(
            df[(df["MetricDate"] == latest_week)
               & (df["PersonId"].isin(ids))
               & df["copilot_active"]]["PersonId"].nunique()
        )),
        power_habitual_latest=("JourneyStage", lambda s: int((s == "Power + Habitual").sum())),
    )
    .reset_index()
)
cohort_latest["active_latest_pct"] = (
    100 * cohort_latest["active_latest"] / cohort_latest["people"]
)
cohort_latest["power_habitual_latest_pct"] = (
    100 * cohort_latest["power_habitual_latest"] / cohort_latest["people"]
)
cohort_latest["weeks_available"] = (
    (latest_week - cohort_latest["first_observed"]).dt.days // 7 + 1
)
cohort_latest["power_habitual_latest_pct_eligible"] = cohort_latest[
    "power_habitual_latest_pct"
].where(cohort_latest["weeks_available"] >= SEGMENT_WINDOW_WEEKS)
save_table(cohort_latest, "03_entry_cohort_journey")


# Use the package-native time-series view and table as the source-of-truth
# presentation of the segment calculation.
native_segment_table = vi.identify_usage_segments(
    seg_input.copy(),
    metric="Total_Copilot_actions_taken",
    version=SEGMENT_VERSION,
    power_thres=POWER_THRESHOLD,
    return_type="table",
).reset_index()
save_table(native_segment_table, "02_native_usage_segments_over_time")

native_segment_fig = vi.identify_usage_segments(
    seg_input.copy(),
    metric="Total_Copilot_actions_taken",
    version=SEGMENT_VERSION,
    power_thres=POWER_THRESHOLD,
    return_type="plot",
)
native_segment_ax = native_segment_fig.axes[0]
native_segment_ax.set_title("", loc="center")
native_segment_ax.set_title(
    "12-week Copilot usage segments over time", loc="left", fontweight="bold"
)
for annotation in native_segment_ax.texts:
    if annotation.get_text().startswith("Usage Segments -"):
        annotation.set_visible(False)
for container in native_segment_ax.containers:
    segment = container.get_label()
    if segment in segment_color_map:
        for patch in container.patches:
            patch.set_facecolor(segment_color_map[segment])
            patch.set_edgecolor("white")
add_subtitle(native_segment_ax, "vi.identify_usage_segments(): 12-week rolling share of PersonId by usage segment")
native_segment_ax.legend(title="Usage Segment", frameon=True)
native_segment_fig.text(
    0.01, -0.01,
    "The first 11 dates have incomplete 12-week histories. Habitual and Power status "
    "still require at least nine active weeks; Novice, Low and Non-user can be assigned "
    "from available history.",
    fontsize=8.5, color=C_GREY,
)
save_figure(native_segment_fig, "02_native_usage_segments")
plt.show()

# Entry cohorts answer a different question from the native segment trend:
# whether newly covered people have had time to activate and form a habit.
plot_cohorts = cohort_latest[cohort_latest["people"] >= MIN_DISPLAY_N].copy()
fig, ax = plt.subplots(figsize=(10.5, 4.8))
ax.bar(plot_cohorts["first_observed"], plot_cohorts["active_latest_pct"],
       width=5.0, color=C_TEAL, label="Active in latest week")
ax.plot(plot_cohorts["first_observed"],
        plot_cohorts["power_habitual_latest_pct_eligible"],
        marker="o", lw=2.2, color=journey_stage_color_map["Power + Habitual"],
        label="Power + Habitual at latest week")
ax.set_ylim(0, 105)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_title("Recent cohorts show lower activation and limited habit tenure",
             loc="left", fontweight="bold")
add_subtitle(ax, "% of each first-observed-week cohort active or Power + Habitual in the latest week")
ax.set_xlabel("First week with Copilot metric coverage")
ax.set_ylabel("% of cohort")
ax.legend(frameon=False)
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
save_figure(fig, "02_entry_cohort_journey")
plt.show()

print(segment_summary.to_string(index=False, formatters={"pct": "{:.1f}%".format}))
print("\nEntry cohorts with at least 30 people:")
print(plot_cohorts.to_string(index=False, formatters={
    "active_latest_pct": "{:.1f}%".format,
    "power_habitual_latest_pct": "{:.1f}%".format,
    "power_habitual_latest_pct_eligible": lambda v: "not yet eligible" if pd.isna(v)
    else f"{v:.1f}%",
}))

## 4. Where adoption is leading or lagging

`vi.create_rank()` provides the package-native organizational comparison. Because usage
segments are categorical, the ranking uses three numeric adoption measures:

- **Power and Habitual Users (%):** the mean of a 0/100 Habitual-or-Power indicator.
- **Active-week share:** the percentage of observed weeks with at least one Copilot action.
- **Average weekly Copilot actions:** usage depth across observed weeks.

The native dumbbell plot shows the highest and lowest qualifying group for each attribute;
the exported tables retain every qualifying group. Function, manager status, and location
are included with a minimum group size of 30. High-cardinality `Organization` remains in
the separate detailed table rather than the leadership-facing plot.

In [ ]:
latest_attributes = (
    df[df["MetricDate"] == latest_week][
        ["PersonId", "FunctionType", "ManagerStatus", "Location", "LevelDesignation"]
    ]
    .drop_duplicates("PersonId")
)
adoption_snapshot = latest_segments.merge(latest_attributes, on="PersonId", how="left")
adoption_snapshot = adoption_snapshot.merge(
    df[df["MetricDate"] == latest_week][
        ["PersonId", "copilot_active", "Total_Copilot_actions_taken"]
    ].drop_duplicates("PersonId"),
    on="PersonId", how="left"
)

manager_summary = (
    adoption_snapshot.groupby("ManagerStatus", dropna=False)
    .agg(
        people=("PersonId", "nunique"),
        active=("copilot_active", "sum"),
        power_habitual=("JourneyStage", lambda s: int((s == "Power + Habitual").sum())),
        median_actions_active=("Total_Copilot_actions_taken",
                               lambda s: s[s > 0].median()),
    )
    .reset_index()
)
manager_summary["active_pct"] = 100 * manager_summary["active"] / manager_summary["people"]
manager_summary["power_habitual_pct"] = (
    100 * manager_summary["power_habitual"] / manager_summary["people"]
)
save_table(manager_summary, "04_manager_adoption")

function_summary = (
    adoption_snapshot.groupby("FunctionType", dropna=False)
    .agg(
        people=("PersonId", "nunique"),
        active=("copilot_active", "sum"),
        power_habitual=("JourneyStage", lambda s: int((s == "Power + Habitual").sum())),
        median_actions_active=("Total_Copilot_actions_taken",
                               lambda s: s[s > 0].median()),
    )
    .reset_index()
)
function_summary["active_pct"] = 100 * function_summary["active"] / function_summary["people"]
function_summary["power_habitual_pct"] = (
    100 * function_summary["power_habitual"] / function_summary["people"]
)
function_summary = (
    function_summary[function_summary["people"] >= MIN_DISPLAY_N]
    .sort_values("active_pct", ascending=False)
    .reset_index(drop=True)
)
save_table(function_summary, "05_function_adoption")

# Where are the Novice Users concentrated? This identifies practical targets for
# targeted conversion (a specific function to target), rather than an abstract
# conversion-rate benchmark that has no obvious next action attached to it.
novice_by_function = (
    adoption_snapshot[adoption_snapshot["UsageSegment_12w"] == "Novice User"]
    .groupby("FunctionType", dropna=False)["PersonId"].nunique()
    .rename("novice_people").reset_index()
    .merge(function_summary[["FunctionType", "people"]], on="FunctionType", how="right")
)
novice_by_function["novice_people"] = novice_by_function["novice_people"].fillna(0).astype(int)
novice_by_function["novice_pct_of_function"] = (
    100 * novice_by_function["novice_people"] / novice_by_function["people"]
)
novice_by_function["share_of_all_novices"] = (
    100 * novice_by_function["novice_people"] / novice_by_function["novice_people"].sum()
)
novice_by_function = novice_by_function.sort_values(
    "novice_people", ascending=False
).reset_index(drop=True)
save_table(novice_by_function, "02c_novice_users_by_function")

segment_manager_mix = pd.crosstab(
    adoption_snapshot["ManagerStatus"],
    adoption_snapshot["UsageSegment_12w"],
    normalize="index",
).mul(100).reindex(columns=segment_order, fill_value=0)


rank_data = (
    person_journey
    .merge(latest_attributes[["PersonId", "FunctionType", "ManagerStatus", "Location"]],
           on="PersonId", how="left")
)
rank_data["MetricDate"] = latest_week
rank_data["Power and Habitual Users (%)"] = (
    rank_data["JourneyStage"] == "Power + Habitual"
).astype(float) * 100
rank_data["Active_week_share_pct"] = rank_data["active_share"] * 100
rank_data["Average_weekly_Copilot_actions"] = (
    rank_data["total_actions"] / rank_data["observed_weeks"]
)
if LOCATION_EXCLUDE:
    rank_data["Location"] = rank_data["Location"].where(
        ~rank_data["Location"].isin(LOCATION_EXCLUDE)
    )

rank_hrvars = ["FunctionType", "ManagerStatus", "Location"]
rank_sustained = vi.create_rank(
    rank_data, metric="Power and Habitual Users (%)", hrvar=rank_hrvars,
    mingroup=MIN_DISPLAY_N, return_type="table",
)
rank_active_share = vi.create_rank(
    rank_data, metric="Active_week_share_pct", hrvar=rank_hrvars,
    mingroup=MIN_DISPLAY_N, return_type="table",
)
rank_action_depth = vi.create_rank(
    rank_data, metric="Average_weekly_Copilot_actions", hrvar=rank_hrvars,
    mingroup=MIN_DISPLAY_N, return_type="table",
)
save_table(rank_sustained, "05a_rank_sustained_adoption")
save_table(rank_active_share, "05b_rank_active_week_share")
save_table(rank_action_depth, "05c_rank_action_depth")

# Preserve the complete table-return outputs, not only the extrema shown in
# create_rank(return_type="plot").
rank_all_measures = pd.concat([
    rank_sustained.assign(measure="Power and Habitual Users (%)"),
    rank_active_share.assign(measure="Active-week share"),
    rank_action_depth.assign(measure="Average weekly Copilot actions"),
], ignore_index=True)
rank_all_measures = rank_all_measures[
    ["measure", "hrvar", "attributes", "metric", "n"]
].rename(columns={
    "hrvar": "organizational_attribute",
    "attributes": "group",
    "metric": "value",
    "n": "people",
})
save_table(rank_all_measures, "05d_create_rank_all_measures")

function_rank = rank_sustained[rank_sustained["hrvar"] == "FunctionType"]
rank_table_view = pd.concat([
    function_rank.head(10),
    function_rank.tail(10),
    rank_sustained[rank_sustained["hrvar"].isin(["ManagerStatus", "Location"])],
]).drop_duplicates(["hrvar", "attributes"]).reset_index(drop=True)
rank_table_view = rank_table_view.rename(columns={
    "hrvar": "Attribute",
    "attributes": "Group",
    "metric": "Power + Habitual Users (%)",
    "n": "People",
})
save_table(rank_table_view, "05e_create_rank_leadership_table")

display(
    rank_table_view.style
    .format({"Power + Habitual Users (%)": "{:.1f}%"})
    .background_gradient(
        subset=["Power + Habitual Users (%)"], cmap="Blues", vmin=0, vmax=100
    )
    .hide(axis="index")
    .set_caption(
        "create_rank(return_type='table'): Power + Habitual adoption by organizational group"
    )
)

rank_fig = vi.create_rank(
    rank_data, metric="Power and Habitual Users (%)", hrvar=rank_hrvars,
    mingroup=MIN_DISPLAY_N, return_type="plot", figsize=(9.5, 5.2),
)
# The package titles the plot from the metric column name; because that name already
# reads cleanly ("Power and Habitual Users (%)"), no override is needed.
rank_ax = rank_fig.axes[0]
for y_pos, hrvar in enumerate(rank_hrvars):
    group_rank = rank_sustained[rank_sustained["hrvar"] == hrvar]
    high = group_rank.iloc[0]
    low = group_rank.iloc[-1]
    rank_ax.annotate(
        f"{low['attributes']} ({low['metric']:.1f}%)",
        (low["metric"], y_pos), xytext=(-6, -18), textcoords="offset points",
        ha="right", fontsize=7.5, color=C_RED,
    )
    rank_ax.annotate(
        f"{high['attributes']} ({high['metric']:.1f}%)",
        (high["metric"], y_pos), xytext=(6, 8), textcoords="offset points",
        ha="left", fontsize=7.5, color=journey_stage_color_map["Power + Habitual"],
    )
add_subtitle(rank_ax, "vi.create_rank(): % of PersonId classified Power or Habitual User in latest week, by group")
save_figure(rank_fig, "03_native_adoption_rank")
plt.show()

# Keep the manager segment mix because it shows the full adoption curve rather
# than only the highest and lowest values returned by the native rank plot.
fig, ax = plt.subplots(figsize=(8.5, 4.8))
segment_manager_mix.plot(
    kind="bar", stacked=True, ax=ax, color=segment_colors, width=0.65
)
ax.set_ylabel("% within manager group")
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_xlabel("")
ax.set_title("Managers are much further along the adoption curve",
             loc="left", fontweight="bold")
add_subtitle(ax, "% of PersonId in each usage segment, by manager status, in the latest week")
ax.legend(frameon=False, fontsize=8, ncol=3)
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
save_figure(fig, "03_manager_segment_mix")
plt.show()

print("\nFull create_rank table outputs are exported in 05d_create_rank_all_measures.csv.")
print("Leadership table view:")
print(rank_table_view.to_string(index=False))

print("\nHighest Power + Habitual adoption groups from create_rank():")
print(rank_sustained.head(12).to_string(index=False))

print("Manager and individual contributor adoption:")
print(manager_summary[["ManagerStatus", "people", "active_pct", "power_habitual_pct",
                       "median_actions_active"]].to_string(index=False, formatters={
    "active_pct": "{:.1f}%".format,
    "power_habitual_pct": "{:.1f}%".format,
    "median_actions_active": "{:.1f}".format,
}))
print("\nFunctions with at least 30 people:")
print(function_summary[["FunctionType", "people", "active_pct", "power_habitual_pct",
                        "median_actions_active"]].to_string(index=False, formatters={
    "active_pct": "{:.1f}%".format,
    "power_habitual_pct": "{:.1f}%".format,
    "median_actions_active": "{:.1f}".format,
}))

### What already works: leading functions as a playbook

Functions that have already crossed a high Power + Habitual adoption threshold are
proof, using this organization's own tooling, policies, and workload, that habitual
use is achievable at scale. Treat them as a replication playbook, not a league table.

In [ ]:
leading_functions = function_summary[
    function_summary["power_habitual_pct"] >= LEADING_FUNCTION_THRESHOLD_PCT
].sort_values("power_habitual_pct", ascending=False).reset_index(drop=True)
save_table(leading_functions, "05f_leading_functions_playbook")

if len(leading_functions):
    fig, ax = plt.subplots(figsize=(9.5, max(3.2, 0.5 * len(leading_functions) + 1.2)))
    bars = ax.barh(leading_functions["FunctionType"], leading_functions["power_habitual_pct"],
                    color=journey_stage_color_map["Power + Habitual"], height=0.6)
    ax.invert_yaxis()
    ax.axvline(LEADING_FUNCTION_THRESHOLD_PCT, color=C_GREY, lw=1, linestyle="--")
    ax.set_xlabel("Power + Habitual Users (%)")
    ax.xaxis.set_major_formatter(mticker.PercentFormatter())
    ax.set_xlim(0, 105)
    ax.set_title("These functions already show adoption can reach scale here",
                 loc="left", fontweight="bold")
    add_subtitle(ax, f"% of PersonId classified Power or Habitual User, by FunctionType, n >= {MIN_DISPLAY_N} per group")
    for bar, value in zip(bars, leading_functions["power_habitual_pct"]):
        ax.annotate(f"{value:.1f}%", (bar.get_width(), bar.get_y() + bar.get_height() / 2),
                    xytext=(6, 0), textcoords="offset points", va="center", fontsize=9)
    ax.grid(axis="y", visible=False)
    fig.tight_layout()
    save_figure(fig, "05f_leading_functions_playbook")
    plt.show()
    print(
        f"{len(leading_functions)} function(s) have already crossed "
        f"{LEADING_FUNCTION_THRESHOLD_PCT:.0f}% Power + Habitual adoption:"
    )
    print(leading_functions[["FunctionType", "people", "power_habitual_pct"]].to_string(
        index=False, formatters={"power_habitual_pct": "{:.1f}%".format}
    ))
else:
    print(
        f"No function has yet crossed {LEADING_FUNCTION_THRESHOLD_PCT:.0f}% Power + "
        "Habitual adoption, so this view is skipped. Lower LEADING_FUNCTION_THRESHOLD_PCT "
        "in the configuration cell to inspect the current leading group instead."
    )

### Sizing the conversion opportunity

A gap becomes a target once it is sized. The scenarios below apply illustrative
conversion rates to the Novice population - including the rate the organization's
own managers already achieve - to show what closing part of the gap would be worth
in headcount terms. These are modelled projections built on stated assumptions, not
measured outcomes.

In [ ]:
novice_n = int(segment_summary.set_index("segment").loc["Novice User", "people"])
total_people = int(segment_summary["people"].sum())
power_habitual_n = int(
    segment_summary.set_index("segment").loc[["Power User", "Habitual User"], "people"].sum()
)
power_habitual_pct_now = pct(power_habitual_n, total_people)

# The manager Power + Habitual rate is used as an internal benchmark: proof that this
# organization's own enablement can already reach that level of sustained use at scale.
manager_rates = manager_summary.set_index("ManagerStatus")["power_habitual_pct"]
manager_rate_pct = float(manager_rates.get("Manager", np.nan))

scenario_rates = list(NOVICE_CONVERSION_RATES)
scenario_labels = [f"{rate:.0%} conversion" for rate in scenario_rates]
if pd.notna(manager_rate_pct):
    scenario_rates.append(manager_rate_pct / 100)
    scenario_labels.append(f"Manager-benchmark conversion ({manager_rate_pct:.1f}%)")

scenario_rows = [{
    "scenario": "Current",
    "conversion_rate_pct": np.nan,
    "additional_people": 0,
    "new_power_habitual_n": power_habitual_n,
    "new_power_habitual_pct": power_habitual_pct_now,
}]
for rate, label in zip(scenario_rates, scenario_labels):
    additional = int(round(rate * novice_n))
    new_n = power_habitual_n + additional
    scenario_rows.append({
        "scenario": label,
        "conversion_rate_pct": 100 * rate,
        "additional_people": additional,
        "new_power_habitual_n": new_n,
        "new_power_habitual_pct": pct(new_n, total_people),
    })
conversion_scenarios = pd.DataFrame(scenario_rows)
save_table(conversion_scenarios, "02b_novice_conversion_scenarios")

fig, ax = plt.subplots(figsize=(9.5, 4.4))
bar_colors = ([C_GREY]
              + [journey_stage_color_map["Power + Habitual"]] * (len(conversion_scenarios) - 1))
bars = ax.barh(conversion_scenarios["scenario"], conversion_scenarios["new_power_habitual_pct"],
                color=bar_colors, height=0.6)
ax.invert_yaxis()
ax.set_xlabel("Power + Habitual Users (% of population)")
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_xlim(0, min(105, conversion_scenarios["new_power_habitual_pct"].max() * 1.25))
ax.set_title("Sizing the novice-conversion opportunity", loc="left", fontweight="bold")
add_subtitle(ax, "Modelled % of population Power + Habitual if a share of Novice Users convert")
for bar, row in zip(bars, conversion_scenarios.itertuples()):
    label = (f"{row.new_power_habitual_pct:.1f}%" if row.additional_people == 0
             else f"{row.new_power_habitual_pct:.1f}% (+{row.additional_people:,})")
    ax.annotate(label, (bar.get_width(), bar.get_y() + bar.get_height() / 2),
                xytext=(6, 0), textcoords="offset points", va="center", fontsize=9)
ax.grid(axis="y", visible=False)
fig.tight_layout()
save_figure(fig, "02b_conversion_scenarios")
plt.show()

# Where the Novice Users actually are: a concrete, actionable view for targeted
# conversion, showing both the count of Novice Users per function and what share of
# the function they represent.
top_novice_functions = novice_by_function[
    novice_by_function["novice_people"] > 0
].head(10)
if len(top_novice_functions):
    fig, ax = plt.subplots(figsize=(9.5, max(3.2, 0.5 * len(top_novice_functions) + 1.2)))
    bars = ax.barh(top_novice_functions["FunctionType"], top_novice_functions["novice_people"],
                    color=segment_color_map["Novice User"], height=0.6)
    ax.invert_yaxis()
    ax.set_xlabel("Novice Users (people)")
    ax.set_title("Novice Users are concentrated in a small number of functions",
                 loc="left", fontweight="bold")
    add_subtitle(ax, f"Count of PersonId classified Novice User, by FunctionType, n >= {MIN_DISPLAY_N} per group")
    ax.set_ylabel("")
    for bar, row in zip(bars, top_novice_functions.itertuples()):
        ax.annotate(f"{row.novice_people:,} ({row.novice_pct_of_function:.0f}% of function)",
                    (bar.get_width(), bar.get_y() + bar.get_height() / 2),
                    xytext=(6, 0), textcoords="offset points", va="center", fontsize=8.5)
    ax.grid(axis="y", visible=False)
    fig.tight_layout()
    save_figure(fig, "02c_novice_users_by_function")
    plt.show()
    print("Top functions by Novice User count (best targets for conversion outreach):")
    print(top_novice_functions[["FunctionType", "novice_people", "novice_pct_of_function",
                                "share_of_all_novices"]].to_string(index=False, formatters={
        "novice_pct_of_function": "{:.1f}%".format,
        "share_of_all_novices": "{:.1f}%".format,
    }))

print(
    f"Novice population: {novice_n:,} people "
    f"({100 * novice_n / total_people:.1f}% of measured population)."
)
print(conversion_scenarios.to_string(index=False, formatters={
    "conversion_rate_pct": lambda v: "-" if pd.isna(v) else f"{v:.1f}%",
    "new_power_habitual_pct": "{:.1f}%".format,
}))

## 5. Collaboration and working-pattern profile by usage segment

`vi.keymetrics_scan()` is used as the primary descriptive comparison across Power,
Habitual, Novice, Low, and Non-user segments.

To keep the comparison like-for-like:

- only people with at least 12 observed Copilot weeks are included;
- each metric is first averaged to one row per person over the trailing 12 weeks;
- an explicit metric list is supplied for reproducibility; and
- the minimum segment size is 30.

The heatmap is normalized **within each metric row**. Colour indicates which segment is
relatively high or low for that metric, not whether the result is inherently favourable.

This query does **not** contain collaboration-network metrics such as internal network
size, external network size, diverse ties, or strong ties. `Collaboration_span` is instead
an hours-based work-session metric: Microsoft defines it as the number of hours spent in
work sessions before, during, and after working hours. It is relabelled below as
**Work session span hours** to avoid implying network breadth. See the
[Microsoft Viva Insights metric reference](https://learn.microsoft.com/en-us/viva/insights/advanced/reference/metrics).

In [ ]:
scan_metrics = [
    "Collaboration_hours",
    "Collaboration_span",
    "Active_connected_hours",
    "Meetings",
    "Meeting_hours",
    "Calls",
    "Call_hours",
    "Chats_sent",
    "Chat_hours",
    "Emails_sent",
    "Email_hours",
    "Multitasking_hours",
    "Available_to_focus_hours",
    "Uninterrupted_hours",
    "Interrupted_hours",
    "After_hours_collaboration_hours",
    "Time_with_leadership",
]
scan_metrics = [metric for metric in scan_metrics if metric in df.columns]

mature_ids = set(
    person_journey.loc[
        person_journey["observed_weeks"] >= SEGMENT_WINDOW_WEEKS, "PersonId"
    ]
)
trailing_start = latest_week - pd.Timedelta(weeks=SEGMENT_WINDOW_WEEKS - 1)
scan_data = (
    df[df["PersonId"].isin(mature_ids) & (df["MetricDate"] >= trailing_start)]
    .groupby("PersonId")[scan_metrics]
    .mean()
    .reset_index()
    .merge(latest_segments[["PersonId", "UsageSegment_12w"]],
           on="PersonId", how="inner")
)
scan_data = scan_data.rename(
    columns={"Collaboration_span": "Work_session_span_hours"}
)
scan_metrics = [
    "Work_session_span_hours" if metric == "Collaboration_span" else metric
    for metric in scan_metrics
]
scan_data["MetricDate"] = latest_week
scan_data["UsageSegment_12w"] = pd.Categorical(
    scan_data["UsageSegment_12w"], categories=segment_order, ordered=True
)

segment_scan_table = vi.keymetrics_scan(
    scan_data,
    hrvar="UsageSegment_12w",
    mingroup=MIN_DISPLAY_N,
    metrics=scan_metrics,
    return_type="table",
)
save_table(segment_scan_table, "06_native_keymetrics_by_segment")

segment_scan_fig = vi.keymetrics_scan(
    scan_data,
    hrvar="UsageSegment_12w",
    mingroup=MIN_DISPLAY_N,
    metrics=scan_metrics,
    return_type="plot",
    low_color="#DCE6EE",
    mid_color="#F4E4BD",
    high_color="#C86B45",
    textsize=8.5,
    plot_row_scaling_factor=0.52,
)
for figure_text in segment_scan_fig.texts:
    if figure_text.get_text().startswith("Data from"):
        figure_text.set_text(
            f"Person-level weekly averages from {trailing_start.date()} to "
            f"{latest_week.date()}; people with at least 12 observed Copilot weeks."
        )
save_figure(segment_scan_fig, "04_native_keymetrics_by_segment")
plt.show()

print(f"Complete-history comparison: {scan_data['PersonId'].nunique():,} people")
print(f"Metrics included: {len(scan_metrics)}")
print(segment_scan_table.to_string(index=False))

## 6. Adjusted and process-level diagnostics

The native key-metrics scan is the primary segment comparison. This section adds two
diagnostics that the package scan does not provide:

1. adjusted sustained-user differences controlling for function and manager status; and
2. robust process ratios for meeting length, after-hours share, meeting multitasking,
   and uninterrupted focus.

These remain observational associations rather than Copilot effects.

In [ ]:
recent_start = latest_week - pd.Timedelta(weeks=RECENT_WEEKS - 1)
recent = observed[observed["MetricDate"] >= recent_start].copy()
recent = recent.merge(latest_segments, on="PersonId", how="inner")

process_source_metrics = [
    "Meetings", "Meeting_hours", "Collaboration_hours",
    "After_hours_collaboration_hours", "Multitasking_hours",
    "Available_to_focus_hours", "Uninterrupted_hours",
]
for metric in process_source_metrics:
    if metric not in recent.columns:
        recent[metric] = np.nan

recent["meeting_length_min"] = np.where(
    recent["Meetings"] > 0, recent["Meeting_hours"] / recent["Meetings"] * 60, np.nan
)
recent["after_hours_share_pct"] = np.where(
    recent["Collaboration_hours"] > 0,
    recent["After_hours_collaboration_hours"] / recent["Collaboration_hours"] * 100,
    np.nan,
)
recent["meeting_multitask_share_pct"] = np.where(
    recent["Meeting_hours"] > 0,
    recent["Multitasking_hours"] / recent["Meeting_hours"] * 100,
    np.nan,
)
recent["focus_realisation_pct"] = np.where(
    recent["Available_to_focus_hours"] > 0,
    recent["Uninterrupted_hours"] / recent["Available_to_focus_hours"] * 100,
    np.nan,
)

ratio_labels = {
    "meeting_length_min": "Meeting length (minutes)",
    "after_hours_share_pct": "After-hours share of collaboration",
    "meeting_multitask_share_pct": "Meeting time spent multitasking",
    "focus_realisation_pct": "Available focus time uninterrupted",
}
ratio_cols = list(ratio_labels)
person_ratios = (
    recent.groupby(["PersonId", "JourneyStage"])[ratio_cols].mean().reset_index()
)
process_summary = (
    person_ratios.groupby("JourneyStage")[ratio_cols].median()
    .reindex(["Power + Habitual", "Emerging", "Non-user"])
    .reset_index()
)
for ratio in ratio_cols:
    if ratio not in process_summary.columns:
        process_summary[ratio] = np.nan
save_table(process_summary, "06_process_ratio_medians")

level_metrics = {
    "Collaboration_hours": "Collaboration hours",
    "Meetings": "Meetings",
    "Meeting_hours": "Meeting hours",
    "Chats_sent": "Chats sent",
    "Emails_sent": "Emails sent",
    "Active_connected_hours": "Active connected hours",
    "Multitasking_hours": "Multitasking hours",
    "Available_to_focus_hours": "Available-to-focus hours",
    "Uninterrupted_hours": "Uninterrupted hours",
    "After_hours_collaboration_hours": "After-hours collaboration",
}
level_metrics = {
    metric: label for metric, label in level_metrics.items()
    if metric in recent.columns and recent[metric].notna().any()
}
person_levels = (
    recent.groupby(["PersonId", "JourneyStage"])[list(level_metrics)].mean().reset_index()
    .merge(latest_attributes[["PersonId", "FunctionType", "ManagerStatus"]],
           on="PersonId", how="left")
)
person_levels["power_habitual"] = (person_levels["JourneyStage"] == "Power + Habitual").astype(int)
person_levels["FunctionType"] = person_levels["FunctionType"].fillna("Missing")
person_levels["ManagerStatus"] = person_levels["ManagerStatus"].fillna("Unknown")

controls = pd.concat([
    person_levels[["power_habitual"]],
    pd.get_dummies(person_levels[["FunctionType", "ManagerStatus"]],
                   drop_first=True, dtype=float),
], axis=1)
controls = sm.add_constant(controls.astype(float))

adjusted_rows = []
for metric, label in level_metrics.items():
    ok = person_levels[metric].notna()
    y = person_levels.loc[ok, metric].astype(float)
    if ok.sum() < 2 * MIN_DISPLAY_N or y.std() == 0:
        continue
    y_z = (y - y.mean()) / y.std()
    model = sm.OLS(y_z, controls.loc[ok]).fit(cov_type="HC3")
    beta = float(model.params["power_habitual"])
    se = float(model.bse["power_habitual"])
    adjusted_rows.append({
        "metric": metric,
        "label": label,
        "n": int(ok.sum()),
        "adjusted_difference_sd": beta,
        "ci_low": beta - 1.96 * se,
        "ci_high": beta + 1.96 * se,
        "p_value": float(model.pvalues["power_habitual"]),
    })

adjusted = pd.DataFrame(adjusted_rows).sort_values("adjusted_difference_sd")
save_table(adjusted, "07_adjusted_working_pattern_associations")

fig, axes = plt.subplots(1, 2, figsize=(14, 5.6))
ypos = np.arange(len(adjusted))
axes[0].errorbar(
    adjusted["adjusted_difference_sd"], ypos,
    xerr=[
        adjusted["adjusted_difference_sd"] - adjusted["ci_low"],
        adjusted["ci_high"] - adjusted["adjusted_difference_sd"],
    ],
    fmt="o", color=journey_stage_color_map["Power + Habitual"], ecolor=C_GREY, capsize=3,
)
axes[0].axvline(0, color=C_TEXT, lw=1)
axes[0].set_yticks(ypos)
axes[0].set_yticklabels(adjusted["label"])
axes[0].set_xlabel("Adjusted difference (standard deviations)")
axes[0].set_title("Power and Habitual Users carry a heavier collaboration load",
                  loc="left", fontweight="bold")
axes[0].grid(axis="y", visible=False)

ratio_base = process_summary.set_index("JourneyStage")
ratio_plot = (
    ratio_base.loc[["Power + Habitual", "Emerging"]]
    .div(ratio_base.loc["Non-user"])
    .sub(1)
    .mul(100)
    .T
)
ratio_plot.index = [ratio_labels[i] for i in ratio_plot.index]
ratio_plot.plot(kind="barh", ax=axes[1],
                color=[journey_stage_color_map["Power + Habitual"],
                       journey_stage_color_map["Emerging"]], width=0.72)
axes[1].axvline(0, color=C_TEXT, lw=1)
axes[1].xaxis.set_major_formatter(mticker.PercentFormatter())
axes[1].set_xlabel("Difference from non-user median")
axes[1].set_title("The process picture is mixed, not uniformly better",
                  loc="left", fontweight="bold")
axes[1].legend(frameon=False)
axes[1].grid(axis="y", visible=False)

fig.tight_layout()
save_figure(fig, "04_working_patterns")
plt.show()

print("Adjusted Power + Habitual associations (function and manager controlled):")
print(adjusted[["label", "n", "adjusted_difference_sd", "ci_low", "ci_high",
                "p_value"]].to_string(index=False, formatters={
    "adjusted_difference_sd": "{:+.3f}".format,
    "ci_low": "{:+.3f}".format,
    "ci_high": "{:+.3f}".format,
    "p_value": "{:.3g}".format,
}))
print("\nMedian process indicators over the latest four weeks:")
print(process_summary.to_string(index=False))

## 7. Same-person check: what changes in heavier Copilot-use weeks?

This view compares each person with themselves and removes common week effects. It controls
for stable individual differences such as role propensity, but it still cannot distinguish
Copilot effects from unusually demanding weeks.

The coefficient is the standard-deviation change in the process indicator associated with
a one-standard-deviation increase in `log(1 + actions)`.

In [ ]:
within_data = observed.copy()
within_data["Total_Copilot_actions_taken"] = (
    within_data["Total_Copilot_actions_taken"].fillna(0)
)
for metric in process_source_metrics:
    if metric not in within_data.columns:
        within_data[metric] = np.nan
within_data["meeting_length_min"] = np.where(
    within_data["Meetings"] > 0,
    within_data["Meeting_hours"] / within_data["Meetings"] * 60, np.nan
)
within_data["after_hours_share_pct"] = np.where(
    within_data["Collaboration_hours"] > 0,
    within_data["After_hours_collaboration_hours"]
    / within_data["Collaboration_hours"] * 100, np.nan
)
within_data["meeting_multitask_share_pct"] = np.where(
    within_data["Meeting_hours"] > 0,
    within_data["Multitasking_hours"] / within_data["Meeting_hours"] * 100, np.nan
)
within_data["focus_realisation_pct"] = np.where(
    within_data["Available_to_focus_hours"] > 0,
    within_data["Uninterrupted_hours"]
    / within_data["Available_to_focus_hours"] * 100, np.nan
)


def two_way_demean(values, person, week, iterations=10):
    result = values.astype(float).copy()
    for _ in range(iterations):
        result = result - result.groupby(person).transform("mean")
        result = result - result.groupby(week).transform("mean")
    return result


within_rows = []
for metric, label in ratio_labels.items():
    work = within_data[[
        "PersonId", "MetricDate", "Total_Copilot_actions_taken", metric
    ]].dropna().copy()
    work["log_actions"] = np.log1p(work["Total_Copilot_actions_taken"])
    work["x_within"] = two_way_demean(
        work["log_actions"], work["PersonId"], work["MetricDate"]
    )
    work["y_within"] = two_way_demean(
        work[metric], work["PersonId"], work["MetricDate"]
    )
    x_sd, y_sd = work["x_within"].std(), work["y_within"].std()
    if len(work) < 2 * MIN_DISPLAY_N or pd.isna(x_sd) or pd.isna(y_sd) or x_sd == 0 or y_sd == 0:
        continue
    model = sm.OLS(
        work["y_within"] / y_sd,
        sm.add_constant(work["x_within"] / x_sd),
    ).fit(cov_type="cluster", cov_kwds={"groups": work["PersonId"]})
    beta = float(model.params["x_within"])
    se = float(model.bse["x_within"])
    within_rows.append({
        "metric": metric,
        "label": label,
        "n_person_weeks": len(work),
        "beta_sd": beta,
        "ci_low": beta - 1.96 * se,
        "ci_high": beta + 1.96 * se,
        "p_value": float(model.pvalues["x_within"]),
    })

within_results = pd.DataFrame(within_rows).sort_values("beta_sd")
save_table(within_results, "08_within_person_process_associations")

fig, ax = plt.subplots(figsize=(9.5, 4.2))
ypos = np.arange(len(within_results))
ax.errorbar(
    within_results["beta_sd"], ypos,
    xerr=[
        within_results["beta_sd"] - within_results["ci_low"],
        within_results["ci_high"] - within_results["beta_sd"],
    ],
    fmt="o", color=C_TEAL, ecolor=C_GREY, capsize=4,
)
ax.axvline(0, color=C_TEXT, lw=1)
ax.set_yticks(ypos)
ax.set_yticklabels(within_results["label"])
ax.set_xlabel("Same-person, week-adjusted association (standard deviations)")
ax.set_title("Heavier-use weeks coincide with less after-hours share and more fragmentation",
             loc="left", fontweight="bold")
ax.grid(axis="y", visible=False)
fig.tight_layout()
save_figure(fig, "05_within_person_process")
plt.show()

print(within_results.to_string(index=False, formatters={
    "beta_sd": "{:+.3f}".format,
    "ci_low": "{:+.3f}".format,
    "ci_high": "{:+.3f}".format,
    "p_value": "{:.3g}".format,
}))

## 8. Evidence-led story synthesis

This section translates the analytical outputs into a reusable evidence matrix for
executive reporting. Each insight records the supporting numbers, the interpretation,
and the principal caveat.

The synthesis deliberately distinguishes **information-exchange intensity** from
information-flow speed or network breadth. This query contains no direct measure of
information velocity and no collaboration-network metrics.

In [ ]:
scan_by_segment = segment_scan_table.set_index("UsageSegment_12w")
process_by_stage = process_summary.set_index("JourneyStage")
within_by_metric = within_results.set_index("metric")
adjusted_by_metric = adjusted.set_index("metric")
segment_counts = segment_summary.set_index("segment")


def relative_gap(metric, group="Power User", reference="Non-user"):
    group_value = float(scan_by_segment.loc[group, metric])
    reference_value = float(scan_by_segment.loc[reference, metric])
    return 100 * (group_value / reference_value - 1)


gap_metrics = {
    "Collaboration_hours": "Collaboration hours",
    "Work_session_span_hours": "Work-session span hours",
    "Active_connected_hours": "Active connected hours",
    "Meetings": "Meetings",
    "Meeting_hours": "Meeting hours",
    "Calls": "Calls",
    "Chats_sent": "Chats sent",
    "Emails_sent": "Emails sent",
    "Multitasking_hours": "Multitasking hours",
    "Available_to_focus_hours": "Available-to-focus hours",
    "Uninterrupted_hours": "Uninterrupted hours",
    "After_hours_collaboration_hours": "After-hours collaboration hours",
    "Time_with_leadership": "Time with leadership",
}
gap_metrics = {
    metric: label for metric, label in gap_metrics.items()
    if metric in scan_by_segment.columns
}

gap_rows = []
for metric, label in gap_metrics.items():
    for group in ["Power User", "Habitual User"]:
        gap_rows.append({
            "metric": metric,
            "label": label,
            "segment": group,
            "segment_value": scan_by_segment.loc[group, metric],
            "non_user_value": scan_by_segment.loc["Non-user", metric],
            "difference_pct": relative_gap(metric, group=group),
        })
segment_gap_table = pd.DataFrame(gap_rows)
save_table(segment_gap_table, "09_segment_gaps_vs_non_users")

sustained_n = int(
    segment_counts.loc["Power User", "people"]
    + segment_counts.loc["Habitual User", "people"]
)
sustained_pct = pct(sustained_n, int(segment_summary["people"].sum()))
novice_pct = float(segment_counts.loc["Novice User", "pct"])

manager_lookup = manager_summary.set_index("ManagerStatus")
manager_sustained = float(manager_lookup["power_habitual_pct"].get("Manager", np.nan))
ic_sustained = float(manager_lookup["power_habitual_pct"].get("IC", np.nan))

function_rank = rank_sustained[rank_sustained["hrvar"] == "FunctionType"]
top_function = function_rank.iloc[0]
bottom_function = function_rank.iloc[-1]

earliest_cohort = cohort_latest.sort_values("first_observed").iloc[0]
latest_cohort = cohort_latest.sort_values("first_observed").iloc[-1]

power_collab = float(scan_by_segment.loc["Power User", "Collaboration_hours"])
non_collab = float(scan_by_segment.loc["Non-user", "Collaboration_hours"])
power_meetings = float(scan_by_segment.loc["Power User", "Meetings"])
non_meetings = float(scan_by_segment.loc["Non-user", "Meetings"])
power_span = float(scan_by_segment.loc["Power User", "Work_session_span_hours"])
non_span = float(scan_by_segment.loc["Non-user", "Work_session_span_hours"])

power_after_hours = float(
    scan_by_segment.loc["Power User", "After_hours_collaboration_hours"]
)
non_after_hours = float(
    scan_by_segment.loc["Non-user", "After_hours_collaboration_hours"]
)
power_after_hours_share = float(
    process_by_stage.loc["Power + Habitual", "after_hours_share_pct"]
)
non_after_hours_share = float(
    process_by_stage.loc["Non-user", "after_hours_share_pct"]
)

power_multitask = float(scan_by_segment.loc["Power User", "Multitasking_hours"])
non_multitask = float(scan_by_segment.loc["Non-user", "Multitasking_hours"])
power_focus = float(scan_by_segment.loc["Power User", "Uninterrupted_hours"]) if "Uninterrupted_hours" in scan_by_segment.columns else np.nan
non_focus = float(scan_by_segment.loc["Non-user", "Uninterrupted_hours"]) if "Uninterrupted_hours" in scan_by_segment.columns else np.nan

insight_evidence = pd.DataFrame([
    {
        "number": 1,
        "theme": "Scale",
        "insight": "Coverage expanded rapidly, while depth among active users remained stable.",
        "evidence": (
            f"Copilot metric coverage increased from {first_row['coverage_pct']:.1f}% "
            f"to {last_row['coverage_pct']:.1f}%; actions per active user moved only "
            f"from {first_row['actions_per_active']:.1f} to "
            f"{last_row['actions_per_active']:.1f}."
        ),
        "implication": "The near-term opportunity is activation and habit formation, not simply adding more covered users.",
        "caveat": "Metric coverage is a proxy because enabled-days data is unavailable.",
    },
    {
        "number": 2,
        "theme": "Habit",
        "insight": "A substantial emerging-user population creates a conversion opportunity.",
        "evidence": (
            f"{sustained_pct:.1f}% are Power or Habitual Users, while "
            f"{novice_pct:.1f}% are Novice Users."
        ),
        "implication": "Targeted use-case reinforcement could convert a large novice pool into repeat users.",
        "caveat": "Recent entrants have incomplete 12-week histories.",
    },
    {
        "number": 3,
        "theme": "Onboarding",
        "insight": "Newly covered cohorts activate materially more slowly.",
        "evidence": (
            f"The earliest cohort is {earliest_cohort['active_latest_pct']:.1f}% active "
            f"in the latest week versus {latest_cohort['active_latest_pct']:.1f}% "
            f"for the newest cohort."
        ),
        "implication": "Onboarding should be measured as a cohort conversion journey rather than a one-time launch.",
        "caveat": "The coverage date may reflect licensing, query scope, or telemetry availability.",
    },
    {
        "number": 4,
        "theme": "Leadership",
        "insight": "Adoption is strongly manager-led.",
        "evidence": (
            f"{manager_sustained:.1f}% of managers are Power or Habitual Users versus "
            f"{ic_sustained:.1f}% of individual contributors."
        ),
        "implication": "Managers can sponsor adoption, but individual-contributor use cases need deliberate reinforcement.",
        "caveat": "Manager roles are structurally more collaboration intensive.",
    },
    {
        "number": 5,
        "theme": "Functional variation",
        "insight": "Power + Habitual adoption differs sharply by function.",
        "evidence": (
            f"{top_function['attributes']} leads qualifying functions at "
            f"{top_function['metric']:.1f}% Power + Habitual adoption versus "
            f"{bottom_function['attributes']} at {bottom_function['metric']:.1f}%."
        ),
        "implication": "The next enablement wave should be role-specific rather than enterprise-generic.",
        "caveat": "These are descriptive group differences, not performance rankings.",
    },
    {
        "number": 6,
        "theme": "Information exchange",
        "insight": "Power Users operate in a substantially more collaboration-intensive environment.",
        "evidence": (
            f"Power Users average {power_collab:.1f} collaboration hours and "
            f"{power_meetings:.1f} meetings weekly versus {non_collab:.1f} hours "
            f"and {non_meetings:.1f} meetings for Non-users."
        ),
        "implication": "Copilot is most embedded where the volume of information exchange is highest.",
        "caveat": "The data measures activity volume, not information-flow speed or quality.",
    },
    {
        "number": 7,
        "theme": "Workday intensity",
        "insight": "Higher usage coincides with longer work-session spans and more connected activity.",
        "evidence": (
            f"Power Users average {power_span:.1f} work-session span hours, "
            f"{relative_gap('Work_session_span_hours'):+.0f}% versus Non-users; "
            f"active connected hours are {relative_gap('Active_connected_hours'):+.0f}% higher."
        ),
        "implication": "Copilot adoption is concentrated in demanding roles and work patterns.",
        "caveat": "Work-session span is an hours metric, not network breadth.",
    },
    {
        "number": 8,
        "theme": "Focus",
        "insight": "Collaboration intensity comes with a focus and fragmentation trade-off.",
        "evidence": (
            f"Power Users average {power_multitask:.1f} multitasking hours versus "
            f"{non_multitask:.1f} for Non-users, and {power_focus:.1f} uninterrupted "
            f"hours versus {non_focus:.1f}."
        ),
        "implication": "Copilot enablement should be paired with meeting, asynchronous-work, and focus-time practices.",
        "caveat": "High-demand weeks may drive both Copilot use and fragmentation.",
    },
    {
        "number": 9,
        "theme": "After-hours",
        "insight": "Higher collaboration does not translate into a proportional after-hours increase.",
        "evidence": (
            f"Power Users record {power_after_hours:.1f} after-hours collaboration hours "
            f"versus {non_after_hours:.1f} for Non-users, but the median after-hours "
            f"share is similar: {power_after_hours_share:.1f}% for Power and Habitual Users "
            f"versus {non_after_hours_share:.1f}% for Non-users."
        ),
        "implication": "The additional collaboration load appears primarily concentrated inside the broader work pattern rather than disproportionately after hours.",
        "caveat": (
            "The raw absolute-hours gap is not statistically distinguishable after "
            f"adjusting for function and manager status "
            f"(beta {adjusted_by_metric.loc['After_hours_collaboration_hours', 'adjusted_difference_sd']:+.3f} SD, "
            f"p={adjusted_by_metric.loc['After_hours_collaboration_hours', 'p_value']:.2f})."
        ),
    },
    {
        "number": 10,
        "theme": "Meetings",
        "insight": "Copilot use is not yet associated with shorter meetings.",
        "evidence": (
            f"Median meeting length is "
            f"{process_by_stage.loc['Power + Habitual', 'meeting_length_min']:.1f} minutes "
            f"for Power and Habitual Users and "
            f"{process_by_stage.loc['Non-user', 'meeting_length_min']:.1f} minutes "
            f"for Non-users; the same-person coefficient is "
            f"{within_by_metric.loc['meeting_length_min', 'beta_sd']:+.3f} SD."
        ),
        "implication": "Meeting recap and asynchronous follow-through have not yet translated into measurable meeting compression.",
        "caveat": "Meeting duration is inferred from weekly meeting hours divided by meeting count.",
    },
])
save_table(insight_evidence, "10_executive_insight_evidence")

print("Top ten evidence-led insights")
print(insight_evidence[["number", "theme", "insight", "evidence"]].to_string(index=False))

## 9. Executive interpretation

The final cell writes an executive summary grounded only in results calculated above.
It leads with the adoption journey, then separates reassuring signals from risks and
recommended next analyses.

In [ ]:
latest_seg = segment_summary.set_index("segment")
latest_manager = manager_summary.set_index("ManagerStatus")
latest_cohorts = cohort_latest.set_index("first_observed")
process = process_summary.set_index("JourneyStage")
scan = segment_scan_table.set_index("UsageSegment_12w")
within = within_results.set_index("metric")

sustained_n = int(
    latest_seg.loc["Power User", "people"]
    + latest_seg.loc["Habitual User", "people"]
)
sustained_pct = pct(sustained_n, int(segment_summary["people"].sum()))

first_date = pd.Timestamp(weekly.iloc[0]["MetricDate"])
latest_date = pd.Timestamp(weekly.iloc[-1]["MetricDate"])
established = cohort_latest.loc[cohort_latest["first_observed"] == first_date].iloc[0]
largest_new = largest_additions.iloc[0]

manager_sustained = float(latest_manager["power_habitual_pct"].get("Manager", np.nan))
ic_sustained = float(latest_manager["power_habitual_pct"].get("IC", np.nan))

summary_lines = [
    "COPILOT ADOPTION AND WAYS-OF-WORKING SUMMARY",
    "=" * 76,
    "",
    "WHAT THE DATA COVERS",
    f"  {df['PersonId'].nunique():,} people across {df['MetricDate'].nunique()} weeks "
    f"({df['MetricDate'].min().date()} to {latest_week.date()}).",
    f"  Removed {blank_rows_removed:,} fully blank trailing rows from the parquet file.",
    "  Copilot enabled-days is absent, so non-null Copilot telemetry is labelled",
    "  metric coverage rather than confirmed licensing.",
    "",
    "ADOPTION JOURNEY",
    f"  Copilot metric coverage rose from {first_row['coverage_pct']:.1f}% to "
    f"{last_row['coverage_pct']:.1f}%. After the baseline week, the largest addition was "
    f"{int(largest_new['newly_observed']):,} people in the week of "
    f"{pd.Timestamp(largest_new['MetricDate']).date()}.",
    f"  Weekly activation among covered people rose from "
    f"{first_row['active_pct_covered']:.1f}% to {last_row['active_pct_covered']:.1f}%, "
    f"while actions per active user stayed broadly stable "
    f"({first_row['actions_per_active']:.1f} to {last_row['actions_per_active']:.1f}).",
    f"  At the latest week, {sustained_n:,} people ({sustained_pct:.1f}%) were "
    "Habitual or Power Users under the 12-week definition.",
    f"  The earliest coverage cohort was {established['active_latest_pct']:.1f}% active "
    f"in the latest week; recent cohorts are materially less activated.",
    "",
    "WHERE THE OPPORTUNITY IS",
    f"  Power and Habitual User status is much more common among managers "
    f"({manager_sustained:.1f}%) "
    f"than individual contributors ({ic_sustained:.1f}%).",
    "  This is both an enablement opportunity and a confounding warning: managers have",
    "  heavier collaboration patterns regardless of Copilot use.",
    "  Function-level activation varies widely. Use the function table to target",
    "  onboarding and role-specific scenarios, not to rank performance.",
    "",
    "NATIVE VIVA INSIGHTS VIEWS",
    f"  The complete-history key-metrics scan compares {scan_data['PersonId'].nunique():,} people across {len(scan_metrics)} collaboration, work-session, and focus metrics.",
    "  The create_rank outputs identify the highest and lowest qualifying groups for Power + Habitual adoption, active-week consistency, and action depth.",
    f"  {rank_sustained[(rank_sustained['hrvar'] == 'FunctionType')].iloc[0]['attributes']} has the highest Power + Habitual adoption among qualifying functions ({rank_sustained[(rank_sustained['hrvar'] == 'FunctionType')].iloc[0]['metric']:.1f}%).",
    f"  Power Users average {scan.loc['Power User', 'Collaboration_hours']:.1f} collaboration hours and {scan.loc['Power User', 'Meetings']:.1f} meetings weekly, compared with {scan.loc['Non-user', 'Collaboration_hours']:.1f} hours and {scan.loc['Non-user', 'Meetings']:.1f} meetings for Non-users.",
    "  Absolute after-hours collaboration is higher for Power Users, but its share of total collaboration remains close across journey stages.",
    "",
    "WAYS OF WORKING",
    "  After adjusting for function and manager status, Power and Habitual Users still show",
    "  materially higher collaboration, meeting, chat, and email activity.",
    "  This indicates that Copilot is concentrated in collaboration-intensive work;",
    "  it does not by itself establish that Copilot created that workload.",
    f"  Median meeting length is similar across journey stages "
    f"({process.loc['Power + Habitual', 'meeting_length_min']:.1f} minutes for Power and Habitual Users "
    f"vs {process.loc['Non-user', 'meeting_length_min']:.1f} for non-users).",
    f"  Median after-hours share is also close "
    f"({process.loc['Power + Habitual', 'after_hours_share_pct']:.1f}% vs "
    f"{process.loc['Non-user', 'after_hours_share_pct']:.1f}%).",
    f"  Power and Habitual Users have a higher meeting multitasking share "
    f"({process.loc['Power + Habitual', 'meeting_multitask_share_pct']:.1f}% vs "
    f"{process.loc['Non-user', 'meeting_multitask_share_pct']:.1f}%) and a lower share "
    f"of available focus time that remains uninterrupted "
    f"({process.loc['Power + Habitual', 'focus_realisation_pct']:.1f}% vs "
    f"{process.loc['Non-user', 'focus_realisation_pct']:.1f}%).",
    "",
    "SAME-PERSON CHECK",
    f"  In heavier Copilot-use weeks, meeting length is essentially unchanged "
    f"(beta {within.loc['meeting_length_min', 'beta_sd']:+.3f} SD).",
    f"  After-hours share is lower "
    f"(beta {within.loc['after_hours_share_pct', 'beta_sd']:+.3f} SD), while meeting "
    f"multitasking is higher "
    f"(beta {within.loc['meeting_multitask_share_pct', 'beta_sd']:+.3f} SD) and focus "
    f"realisation is lower "
    f"(beta {within.loc['focus_realisation_pct', 'beta_sd']:+.3f} SD).",
    "  The most defensible interpretation is that Copilot is being used in demanding,",
    "  fragmented weeks. The after-hours result is reassuring, but the focus signal",
    "  suggests value will depend on pairing Copilot with better collaboration practices.",
    "",
    "SIZING THE OPPORTUNITY",
    f"  Converting Novice Users at the manager-benchmark rate ({manager_rate_pct:.1f}%) would "
    f"add {int(conversion_scenarios.iloc[-1]['additional_people']):,} Power or Habitual Users, "
    f"taking the organization to {conversion_scenarios.iloc[-1]['new_power_habitual_pct']:.1f}%.",
    (
        f"  {len(leading_functions)} function(s) already exceed "
        f"{LEADING_FUNCTION_THRESHOLD_PCT:.0f}% Power + Habitual adoption and are candidates "
        "for a replication playbook."
        if len(leading_functions) else
        f"  No function has yet crossed {LEADING_FUNCTION_THRESHOLD_PCT:.0f}% Power + Habitual "
        "adoption."
    ),
    "",
    "RECOMMENDED ACTIONS",
    "  1. Confirm whether the June/July coverage jumps reflect licensing, query scope,",
    "     telemetry changes, or a deliberate enablement wave.",
    "  2. Prioritise recent coverage cohorts and lower-adoption functions for role-based",
    "     onboarding, then track 12-week conversion to Habitual or Power status.",
    "  3. Use managers as adoption sponsors, while building explicit individual-",
    "     contributor use cases so adoption does not remain manager-led.",
    "  4. Pair Copilot enablement with meeting recap, asynchronous updates, and focus-time",
    "     norms; monitor multitasking share and focus realisation as guardrail metrics.",
    "  5. Re-run with at least 26 weeks of data and a populated seniority attribute before",
    "     making longer-horizon retention claims. Causal claims require a separate design.",
    "",
    "INTERPRETATION LIMIT",
    "  All findings are observational associations. They do not prove Copilot caused",
    "  changes in collaboration, focus, or wellbeing.",
]

summary_text = "\n".join(summary_lines)
print(summary_text)
(OUTPUT_DIR / "executive_summary.txt").write_text(summary_text, encoding="utf-8")

print(f"\nWrote {len(TABLES)} tables, {len(FIGURES)} figures, and executive_summary.txt")
print(f"Output directory: {OUTPUT_DIR.resolve()}")